# E3-E1 — Audit d'équité des entrées et des cibles

**But :** vérifier, avant toute nouvelle expérience, que les pipelines utilisent les mêmes informations lorsqu'une comparaison est présentée comme « à entrées identiques ». Cet audit ne réentraîne aucun modèle et ne modifie aucun résultat historique.

- **C1, entrée locale** : Ridge, RF, XGBoost, LSTM, TCN et GCN natif reçoivent, pour chaque cible, uniquement son historique de **5 × 3** observations. Pour le GCN natif, l'adjacence identité doit être vérifiée et la sortie d'un nœud doit être indépendante des autres nœuds.
- **C2, entrée multidevices** : un GCN relationnel reçoit **7 × 5 × 3** observations. Les modèles non graphiques **n'ont pas encore** été entraînés avec cette même information ; la comparaison C2 ne peut donc pas être déclarée équitable avant l'expérience complémentaire.
- Les mêmes clés de test, cibles et timestamps doivent être utilisés aux quatre horizons.
- L'identité des **informations** ne garantit pas l'identité des capacités des architectures ou de la procédure d'optimisation.

**Correctif V2 :** les tenseurs graphiques ont la disposition `[snapshots, 5, 7, 3]` constatée sur le jeu de données. La comparaison aux tenseurs séquentiels utilise `transpose(0,2,1,3)`.

**Important :** cet audit exploite les fichiers de `E3_07ABCD_multihorizon_aligned` et les prédictions déjà générées. Il s'arrête sur toute incohérence structurante, au lieu de déclarer la validation réussie par défaut.


In [ ]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd

ROOT=Path.cwd().resolve()
REP=ROOT/'outputs/E3_aligned_representations'
COMMON=ROOT/'outputs/E3_common_evaluation'
OUT=ROOT/'outputs/E3_E1_input_fairness_audit'
HORIZONS=[15,30,60,120]
SPLITS=['train','validation','test']
KEYS=['uuid','timestamp','split']
EXPECTED={'train':64393,'validation':13727,'test':13643}
assert REP.is_dir() and COMMON.is_dir(), 'Missing E3 input folders.'
OUT.mkdir(parents=True,exist_ok=True)
common=pd.read_csv(COMMON/'01_common_keys_all_horizons.csv')
common['timestamp']=pd.to_datetime(common['timestamp'],utc=True)
assert len(common)==sum(EXPECTED.values())
assert common.groupby('split').size().to_dict()==EXPECTED
assert not common.duplicated(KEYS).any()
checks=[]
def check(h,split,subject,condition,detail=''):
    checks.append({'horizon_min':h,'split':split,'check':subject,
                   'PASS':bool(condition),'detail':str(detail)})
    if not condition:
        raise AssertionError(f'FAIL h={h}, split={split}: {subject}: {detail}')
print('Project:',ROOT)


In [ ]:
# Inspect actual E3 source tensor layouts and node ordering.
for h in HORIZONS:
    folder=REP/f'h{h:03d}'
    node_index=pd.read_csv(folder/'01_graph_node_index.csv').sort_values('node_index')
    devices=node_index['uuid'].astype(str).tolist()
    check(h,'all','seven distinct indexed devices',
          len(devices)==7 and len(set(devices))==7
          and node_index.node_index.tolist()==list(range(7)))
    for split in SPLITS:
        meta=pd.read_csv(folder/f'metadata_{split}.csv')
        for col in ('timestamp','target_timestamp'):
            meta[col]=pd.to_datetime(meta[col],utc=True)
        xs=np.load(folder/f'X_sequence_{split}_unscaled.npy',mmap_mode='r')
        xss=np.load(folder/f'X_sequence_{split}_scaled.npy',mmap_mode='r')
        xt=np.load(folder/f'X_tabular_{split}_unscaled.npy',mmap_mode='r')
        xts=np.load(folder/f'X_tabular_{split}_scaled.npy',mmap_mode='r')
        xg=np.load(folder/f'X_graph_{split}_unscaled.npy',mmap_mode='r')
        xgs=np.load(folder/f'X_graph_{split}_scaled.npy',mmap_mode='r')
        ys=np.load(folder/f'y_sequence_{split}.npy',mmap_mode='r')
        yt=np.load(folder/f'y_tabular_{split}.npy',mmap_mode='r')
        yg=np.load(folder/f'y_graph_{split}.npy',mmap_mode='r')
        idx=np.load(COMMON/f'indices_h{h:03d}_{split}.npy')
        N=len(meta);S=N//7
        check(h,split,'tensor shapes',
              N%7==0 and xs.shape==(N,5,3) and xss.shape==(N,5,3)
              and xt.shape==(N,15) and xts.shape==(N,15)
              and xg.shape==(S,5,7,3) and xgs.shape==(S,5,7,3)
              and ys.shape==(N,) and yt.shape==(N,) and yg.shape==(S,7),
              f'metadata={N}, graph={xg.shape}, seq={xs.shape}')
        check(h,split,'node ordering',
              meta.uuid.astype(str).to_numpy().reshape(-1,7).tolist()
              == [devices]*S)
        check(h,split,'target time offset',
              (meta.target_timestamp-meta.timestamp).eq(pd.Timedelta(minutes=h)).all())
        check(h,split,'same targets',
              np.array_equal(ys,yt) and np.array_equal(ys,yg.reshape(-1))
              and np.allclose(ys,meta.target.to_numpy(),rtol=0,atol=1e-5))
        check(h,split,'identical local inputs tabular/sequence/graph',
              np.array_equal(xt,xs.reshape(-1,15))
              and np.array_equal(xg.transpose(0,2,1,3).reshape(-1,5,3),xs))
        check(h,split,'identical scaled inputs tabular/sequence/graph',
              np.array_equal(xts,xss.reshape(-1,15))
              and np.array_equal(xgs.transpose(0,2,1,3).reshape(-1,5,3),xss))
        check(h,split,'common indices valid',
              len(idx)==EXPECTED[split] and len(set(idx.tolist()))==len(idx)
              and idx.min()>=0 and idx.max()<N)
        got=meta.iloc[idx][KEYS].sort_values(KEYS).reset_index(drop=True)
        exp=common.loc[common.split.eq(split),KEYS].sort_values(KEYS).reset_index(drop=True)
        check(h,split,'identical common keys',got.equals(exp))
        # Each target node has its own history and the same 15 features as tabular/sequence.
        # This proves equality of data values, not independence of a trained graph model.
        check(h,split,'all values finite',
              np.isfinite(xs).all() and np.isfinite(xss).all()
              and np.isfinite(ys).all())
print(pd.DataFrame(checks).groupby('check')['PASS'].agg(['sum','count']))


In [ ]:
# Verify GCN native adjacency is identity and prove node-local dependence
# for the *actual architecture* used by E3_10D (GraphConv with einsum).
# This is a mathematical and synthetic forward-pass check; not a weight audit.
import torch
from torch import nn
class GraphConv(nn.Module):
    def __init__(self,fin,fout):
        super().__init__()
        self.lin=nn.Linear(fin,fout)
    def forward(self,x,A):
        return self.lin(torch.einsum('ij,bjf->bif',A,x))
class TemporalGCN(nn.Module):
    def __init__(self,hidden=32):
        super().__init__()
        self.gc1=GraphConv(15,hidden)
        self.gc2=GraphConv(hidden,hidden)
        self.head=nn.Linear(hidden,1)
    def forward(self,x,A):
        z=x.permute(0,2,1,3).contiguous().reshape(x.shape[0],7,15)
        z=torch.relu(self.gc1(z,A))
        z=torch.relu(self.gc2(z,A))
        return self.head(z).squeeze(-1)
torch.manual_seed(42)
model=TemporalGCN(16).eval()
A=torch.eye(7)
x=torch.randn(2,5,7,3)
with torch.no_grad():
    baseline=model(x,A)
    changed=x.clone();changed[:,:,1:,:]+=100.0
    after=model(changed,A)
    err=float((baseline[:,0]-after[:,0]).abs().max())
check(0,'synthetic','GCN native adjacency = I and no cross-node forward influence',
      err<1e-7,f'node0 max absolute difference={err}')
print('GCN-native node-locality test: PASS')


In [ ]:
# Check which E3 result sets are present, without assuming they were produced.
# Explicitly separate the source-level proof from an audit of fitted model predictions.
sources={
 'tabular_A':ROOT/'outputs/E3_10A_persistence_ridge_multihorizon',
 'tabular_B':ROOT/'outputs/E3_10B_RF_XGBOOST_multihorizon',
 'sequential_C':ROOT/'outputs/E3_10C_LSTM_TCN_multihorizon',
 'graph_D':ROOT/'outputs/E3_10D_GCN_multihorizon'
}
presence=[]
for name,folder in sources.items():
    exists=folder.is_dir()
    prediction_files=sorted(p.name for p in folder.glob('*predictions*.csv')) if exists else []
    presence.append({'experiment':name,'directory_exists':exists,
                     'prediction_files':'; '.join(prediction_files)})
presence=pd.DataFrame(presence)
print(presence.to_string(index=False))


In [ ]:
# Final report: C1 information-level fairness and C2 pending work.
report=pd.DataFrame(checks)
report.to_csv(OUT/'01_tensor_and_key_checks.csv',index=False)
presence.to_csv(OUT/'02_existing_predictions_inventory.csv',index=False)
decision=pd.DataFrame([
 {'comparison':'C1 local input','models':'Ridge/RF/XGB/LSTM/TCN/GCN native',
  'information_per_target':'5 × 3 (15 values)',
  'status':'SOURCE INPUTS VERIFIED; confirm fitted model implementation',
  'caveat':'GCN-native shared weights across devices; different inductive bias'},
 {'comparison':'C2 multi-device input','models':'Ridge/RF/XGB/LSTM/TCN/GCN operational/shuffled',
  'information_per_target':'7 × 5 × 3 (105 values)',
  'status':'NOT YET COMPARABLE',
  'caveat':'Non-graph multi-device baselines require training'},
 {'comparison':'C3 graph topology','models':'GCN native/operational/shuffled',
  'information_per_target':'same 7 × 5 × 3 tensor, but identity adjacency blocks neighbors',
  'status':'SOURCE INPUTS VERIFIED; architecture/convergence still to audit',
  'caveat':'Different information reachability across topologies'}
])
decision.to_csv(OUT/'03_comparison_fairness_decisions.csv',index=False)
protocol={'source':'E3_aligned_representations','common_test_nodes':13643,
          'horizons_minutes':HORIZONS,'local_features':15,
          'multi_device_features':105,
          'scope':'source tensors, target keys, graph-native independence',
          'not_verified':'trained checkpoints or historical 08C topology equivalence',
          'historical_results':'unchanged'}
(OUT/'04_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
assert report.PASS.all()
print('AUDIT PASSED:',len(report),'source checks')
print('Saved:',OUT)
print(decision.to_string(index=False))


## Interprétation

**C1 est admissible comme comparaison à information locale identique** si les codes d'entraînement sont cohérents avec les fichiers audités. Les paramètres partagés du GCN natif entre appareils doivent être explicitement signalés.

**C2 n'est pas encore réalisée.** Prochaine expérience : entraînement de baselines non graphiques multidevices à entrée `7 × 5 × 3`, sortie `7` températures, avec même découpage, mêmes horizons et mêmes clés communes. Pour isoler davantage l'effet des relations, ajouter une ablation GCN sans mélange et une variante relationnelle à paramètres comparables.

**Attention au format :** `X_graph` est stocké en `[snapshots, 5, 7, 3]` d’après les dimensions observées sur le jeu à 15 minutes. Le présent audit vérifie ce format aux quatre horizons. Il transpose explicitement les axes pour comparer les historiques nodaux. Vérifier aussi que tout entraînement GCN utilise ce même ordre d’axes.
